# HandPose Accuracy v3 · Mục tiêu PCK@0.05 ≥ 90%

**Đây là cấu hình mới để huấn luyện và đo mục tiêu, chưa phải model đã đạt 90%.**
Metric giữ nguyên: ảnh đầu vào 128×128, đúng khi sai số ≤ 6,4 px; mọi khớp hợp lệ đều được tính.
Không bỏ khớp bị che, không tăng bán kính đúng, không dùng test để chọn checkpoint.

Kiến trúc mới: **MobileNetV2 pretrained ImageNet + decoder nhiều mức có skip connections**, heatmap **64×64**.
Dùng toàn bộ 4 biến thể nền mỗi epoch, augmentation hình học đồng bộ ảnh–nhãn, heatmap CE + coordinate loss,
checkpoint chọn theo validation PCK, lưu trạng thái để resume khi Kaggle hết phiên.

**Cách chạy:** Kaggle GPU + Internet → thêm FreiHAND `danieldelro/freihand` → Run All.
Lần đầu `MODE="train"`. Sau khi ngắt phiên, upload `handpose_accuracy_last.pth`, đặt `MODE="resume"`
và `CHECKPOINT` đúng đường dẫn. Muốn chỉ export/eval: `MODE="reuse"` với `handpose_accuracy_best.pth`.
**Checkpoint TinyHandPose cũ không tương thích kiến trúc mới.**

Backbone bỏ classification head và tầng 1280 kênh; decoder dùng depthwise convolution để giảm chi phí trên BBB. Notebook có xuất NCNN và kiểm tra INT8, nhưng **chưa xác nhận
FPS/RAM trên BBB**. Nếu model đạt chất lượng nhưng quá chậm trên BBB, bước sau là distillation sang student nhỏ
và/hoặc QAT; notebook này không giả định các bước đó đã hoàn tất.

`TARGET_PCK=0.90` là ngưỡng tuyệt đối. Chỉ đưa INT8 vào `approved_models` nếu cả validation và test đạt ≥90%
và mức giảm so với FP32 nằm trong giới hạn. Nếu chưa đạt, báo cáo ghi rõ **NOT_REACHED**.

### Đối chiếu log bạn gửi ngày 11/10
Log v2: 70.077 tham số, test PCK=0,5896234 (58,96%), EPE=7,78 pixel.
Không thấy kết quả PCK 77–78% trong log đó. `passed` ở v2 chỉ kiểm tra mức suy giảm INT8.
V3 bên dưới thay kiến trúc và quy trình train; giữ nguyên định nghĩa PCK.


## 1. Cấu hình và môi trường


In [ ]:
from pathlib import Path
CFG = {
    "DATA_ROOT": None, "OUTPUT_BASE": "/kaggle/working",
    "MODE": "train",  # train / resume / reuse
    "CHECKPOINT": None,
    "PRETRAINED": True,  # tải MobileNetV2 ImageNet khi train mới
    "TARGET_PCK": 0.90,
    "SEED": 42, "BATCH_SIZE": 32, "NUM_WORKERS": 2,
    "EPOCHS": 120, "PATIENCE": 25, "MIN_EPOCHS": 40,
    "LR": 3e-4, "WEIGHT_DECAY": 1e-4, "WARMUP_EPOCHS": 5,
    "SIGMA": 2.0, "COORD_LOSS_WEIGHT": 5.0,
    "TRAIN_FRACTION": 0.80, "VAL_FRACTION": 0.10,
    "EXPECTED_BASE_SAMPLES": 32560, "CALIBRATION_SAMPLES": 256,
    "LOG_EVERY_BATCHES": 200, "EVAL_BATCH_SIZE": 16,
    "CPU_THREADS": 2, "MAX_EPE_DELTA": 1.0, "MAX_PCK_DROP": 0.02,
    "BUILD_NCNN": True, "NCNN_REF": "master", "NCNN_METHODS": ["kl", "aciq"], "BUILD_JOBS": 2,
    "AUG_ROTATION": 30.0, "AUG_SCALE": [0.85, 1.15], "AUG_SHIFT": 0.03,
    "RUN_OVERFIT_CHECK": True, "OVERFIT_STEPS": 250,
}
assert CFG["MODE"] in {"train", "resume", "reuse"}
assert CFG["EPOCHS"] > CFG["WARMUP_EPOCHS"]


In [ ]:
from pathlib import Path
MODEL_SOURCE = 'from __future__ import annotations\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torchvision.models import mobilenet_v2, MobileNet_V2_Weights\nIMAGE_SIZE = 128\nHEATMAP_SIZE = 64\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\nFINGERS = {\n    "thumb": [1, 2, 3, 4],\n    "index": [5, 6, 7, 8],\n    "middle": [9, 10, 11, 12],\n    "ring": [13, 14, 15, 16],\n    "pinky": [17, 18, 19, 20],\n}\n\n\n\nclass ConvBNReLU(nn.Sequential):\n    def __init__(self, cin, cout, kernel=3, groups=1):\n        super().__init__(nn.Conv2d(cin, cout, kernel, padding=kernel//2, groups=groups, bias=False),\n                         nn.BatchNorm2d(cout), nn.ReLU(inplace=False))\nclass Refine(nn.Sequential):\n    def __init__(self, channels):\n        super().__init__(ConvBNReLU(channels, channels, groups=channels),\n                         ConvBNReLU(channels, channels, kernel=1),\n                         ConvBNReLU(channels, channels, groups=channels),\n                         ConvBNReLU(channels, channels, kernel=1))\nclass HandPoseAccuracy(nn.Module):\n    def __init__(self, pretrained=False):\n        super().__init__()\n        features = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT if pretrained else None).features\n        self.stage64 = nn.Sequential(*[features[i] for i in range(0,2)])\n        self.stage32 = nn.Sequential(*[features[i] for i in range(2,4)])\n        self.stage16 = nn.Sequential(*[features[i] for i in range(4,7)])\n        self.stage8 = nn.Sequential(*[features[i] for i in range(7,14)])\n        self.stage4 = nn.Sequential(*[features[i] for i in range(14,18)])\n        self.top = ConvBNReLU(320, 64, kernel=1)\n        self.lat8 = ConvBNReLU(96,64,kernel=1)\n        self.ref8 = Refine(64)\n        self.lat16 = ConvBNReLU(32,64,kernel=1)\n        self.ref16 = Refine(64)\n        self.reduce = ConvBNReLU(64,48,kernel=1)\n        self.lat32 = ConvBNReLU(24,48,kernel=1)\n        self.ref32 = Refine(48)\n        self.up64 = ConvBNReLU(48,32,kernel=1)\n        self.lat64 = ConvBNReLU(16,32,kernel=1)\n        self.ref64 = Refine(32)\n        self.output = nn.Conv2d(32,21,1)\n    def forward(self,x):\n        s64 = self.stage64(x)\n        s32 = self.stage32(s64)\n        s16 = self.stage16(s32)\n        s8 = self.stage8(s16)\n        s4 = self.stage4(s8)\n        x = self.ref8(F.interpolate(self.top(s4),scale_factor=2.,mode=\'nearest\')+self.lat8(s8))\n        x = self.ref16(F.interpolate(x,scale_factor=2.,mode=\'nearest\')+self.lat16(s16))\n        x = self.ref32(F.interpolate(self.reduce(x),scale_factor=2.,mode=\'nearest\')+self.lat32(s32))\n        x = self.ref64(F.interpolate(self.up64(x),scale_factor=2.,mode=\'nearest\')+self.lat64(s64))\n        return self.output(x)\ndef decode_heatmaps(logits):\n    b,j,h,w=logits.shape\n    index=logits.flatten(2).argmax(-1)\n    return torch.stack(((index.remainder(w).float()+0.5)/w,\n                        (index.div(w,rounding_mode=\'floor\').float()+0.5)/h),dim=-1)\ndef model_summary():\n    m=HandPoseAccuracy(pretrained=False)\n    return {\'parameters\':sum(p.numel() for p in m.parameters()),\'input_size\':IMAGE_SIZE,\'heatmap_size\':HEATMAP_SIZE}\n'
Path("handpose_accuracy.py").write_text(MODEL_SOURCE,encoding="utf-8")


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm


import copy
import importlib.metadata
import traceback
import zipfile
from torch.utils.data import Subset
from IPython.display import display, FileLink
from handpose_accuracy import (
    FINGERS, HEATMAP_SIZE, IMAGE_SIZE, MEAN, STD,
    HandPoseAccuracy, decode_heatmaps, model_summary,
)

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.set_num_threads(CFG["CPU_THREADS"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(CFG["SEED"])
RUN_DIR = Path(CFG["OUTPUT_BASE"]) / ("handpose_accuracy_v3_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
REPORT_DIR = RUN_DIR / "report"
REPORT_DIR.mkdir()
(RUN_DIR / "handpose_accuracy.py").write_text(MODEL_SOURCE, encoding="utf-8")

def write_json(path, value):
    def default(obj):
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, np.generic): return obj.item()
        if isinstance(obj, Path): return str(obj)
        raise TypeError(type(obj).__name__)
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, default=default), encoding="utf-8")

def run_checked(args, cwd=None, log_name="commands.log"):
    print("Running:", " ".join(map(str, args)), flush=True)
    log = REPORT_DIR / log_name
    with log.open("a") as f:
        f.write("\n$ " + " ".join(map(str, args)) + "\n")
        f.flush()
        result = subprocess.run(list(map(str, args)), cwd=cwd, stdout=f, stderr=subprocess.STDOUT)
    if result.returncode:
        print(log.read_text(errors="replace")[-8000:])
        raise RuntimeError(f"Command failed ({result.returncode}); see {log}")

def resolve_checkpoint():
    if CFG["MODE"] == "train": return None
    if not CFG["CHECKPOINT"]:
        raise ValueError("Đặt CHECKPOINT rõ ràng khi resume/reuse; checkpoint kiến trúc cũ không dùng được.")
    p = Path(CFG["CHECKPOINT"])
    if not p.is_file(): raise FileNotFoundError(p)
    return p

SOURCE_CHECKPOINT = resolve_checkpoint()
print("Python:", platform.python_version(), "Torch:", torch.__version__)
print("Device:", DEVICE, "Model:", model_summary())
print("Checkpoint:", SOURCE_CHECKPOINT or "Train mới", "Output:", RUN_DIR)
write_json(RUN_DIR / "config.json", CFG)

# Kiểm tra giao diện và đếm chi phí convolution trước khi train.
profile_model=HandPoseAccuracy(pretrained=False).cpu().eval()
conv_macs=[0]
def count_conv(module, inputs, output):
    conv_macs[0] += int(output.numel()*(module.in_channels//module.groups)*module.kernel_size[0]*module.kernel_size[1])
hooks=[m.register_forward_hook(count_conv) for m in profile_model.modules() if isinstance(m,nn.Conv2d)]
with torch.inference_mode():
    profile_output=profile_model(torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE))
assert tuple(profile_output.shape)==(1,21,HEATMAP_SIZE,HEATMAP_SIZE)
architecture_profile={"parameters":sum(p.numel() for p in profile_model.parameters()),
                      "conv_macs_per_crop":conv_macs[0],"input_size":IMAGE_SIZE,
                      "heatmap_size":HEATMAP_SIZE,"bbb_measured":False}
for hook in hooks: hook.remove()
print("Architecture profile (không phải FPS đo trên BBB):",architecture_profile)
write_json(RUN_DIR/"architecture_profile.json",architecture_profile)
del profile_model,profile_output,hooks


## 2. FreiHAND và chia tập theo base ID
Giữ seed và tỷ lệ chia giống notebook cũ. Bốn biến thể của một bàn tay luôn ở cùng tập.


In [ ]:
def find_freihand(root: str | None = None) -> Path:
    base = Path(root) if root else Path("/kaggle/input")
    candidates = [path.parent for path in base.rglob("training_xyz.json")]
    if (base / "training_xyz.json").is_file():
        candidates.append(base)
    candidates = sorted(
        {
            path.resolve()
            for path in candidates
            if (path / "training_K.json").is_file()
            and (path / "training/rgb").is_dir()
        }
    )
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one FreiHAND dataset, found {len(candidates)}: {candidates}"
        )
    return candidates[0]


ROOT = find_freihand(CFG["DATA_ROOT"])
print("FreiHAND root:", ROOT, flush=True)
BASE_N = CFG["EXPECTED_BASE_SAMPLES"]
K = np.asarray(json.loads((ROOT / "training_K.json").read_text()), dtype=np.float32)
XYZ = np.asarray(json.loads((ROOT / "training_xyz.json").read_text()), dtype=np.float32)
if K.shape != (BASE_N, 3, 3) or XYZ.shape != (BASE_N, 21, 3):
    raise ValueError(f"Unexpected FreiHAND labels: K={K.shape}, XYZ={XYZ.shape}")

projected = np.einsum("nij,nkj->nki", K, XYZ)
UV = projected[:, :, :2] / projected[:, :, 2:3]

images_by_base: dict[int, list[str]] = {}
for path in sorted((ROOT / "training/rgb").glob("*.jpg")):
    if not path.stem.isdigit():
        continue
    image_id = int(path.stem)
    if image_id >= 4 * BASE_N:
        raise ValueError(f"Image id outside FreiHAND convention: {path.name}")
    images_by_base.setdefault(image_id % BASE_N, []).append(str(path))
if len(images_by_base) != BASE_N:
    raise ValueError(f"Only found {len(images_by_base)}/{BASE_N} base samples")
print(f"Indexed {sum(map(len, images_by_base.values())):,} images", flush=True)

base_ids = np.arange(BASE_N)
np.random.default_rng(CFG["SEED"]).shuffle(base_ids)
cut1 = int(BASE_N * CFG["TRAIN_FRACTION"])
cut2 = cut1 + int(BASE_N * CFG["VAL_FRACTION"])
SPLIT_IDS = {
    "train": base_ids[:cut1],
    "val": base_ids[cut1:cut2],
    "test": base_ids[cut2:],
}
print(
    "Split base samples:",
    {name: len(ids) for name, ids in SPLIT_IDS.items()},
    flush=True,
)
dataset_summary = pd.DataFrame(
    [
        {
            "split": name,
            "base_samples": len(ids),
            "image_variants": sum(len(images_by_base[int(base_id)]) for base_id in ids),
        }
        for name, ids in SPLIT_IDS.items()
    ]
)
dataset_summary.to_csv(REPORT_DIR / "dataset_summary.csv", index=False)
assert not (set(SPLIT_IDS["train"]) & set(SPLIT_IDS["val"]))
assert not (set(SPLIT_IDS["train"]) & set(SPLIT_IDS["test"]))
assert not (set(SPLIT_IDS["val"]) & set(SPLIT_IDS["test"]))
np.savez(RUN_DIR / "split_ids.npz", **SPLIT_IDS)


In [ ]:
def make_transform(training):
    operations = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
    if training:
        operations.append(transforms.ColorJitter(brightness=.2, contrast=.2, saturation=.15, hue=.03))
    operations += [transforms.ToTensor(), transforms.Normalize(MEAN,STD)]
    return transforms.Compose(operations)

def augment_geometry(image, points):
    width,height=image.size
    angle=np.deg2rad(random.uniform(-CFG["AUG_ROTATION"],CFG["AUG_ROTATION"]))
    scale=random.uniform(*CFG["AUG_SCALE"])
    rot=scale*np.array([[np.cos(angle),-np.sin(angle)],[np.sin(angle),np.cos(angle)]],np.float32)
    center=np.array([width,height],np.float32)/2
    shift=np.array([random.uniform(-1,1)*width,random.uniform(-1,1)*height])*CFG["AUG_SHIFT"]
    offset=center+shift-rot@center
    forward=np.eye(3); forward[:2,:2]=rot; forward[:2,2]=offset
    inverse=np.linalg.inv(forward)
    image=image.transform(image.size,Image.Transform.AFFINE,tuple(inverse[:2].reshape(-1)),
                          resample=Image.Resampling.BILINEAR,fillcolor=(124,116,104))
    pixels=points*np.array([width,height])
    points=(pixels@rot.T+offset)/np.array([width,height])
    if random.random()<.5:
        image=ImageOps.mirror(image)
        points[:,0]=1-points[:,0]
    return image,points.astype(np.float32)

class FreiHandTiny(Dataset):
    def __init__(self, ids, training):
        self.training=training
        self.items=[(int(base_id),path) for base_id in ids for path in images_by_base[int(base_id)]]
        self.transform=make_transform(training)
    def __len__(self): return len(self.items)
    def __getitem__(self,index):
        base_id,path=self.items[index]
        with Image.open(path) as source: image=source.convert("RGB")
        w,h=image.size
        if (w,h)!=(224,224): raise ValueError(f"Unexpected size: {path}")
        points=(UV[base_id].copy()+.5)/np.array([w,h],np.float32)
        if self.training: image,points=augment_geometry(image,points)
        valid=np.isfinite(points).all(1)&(points>=0).all(1)&(points<1).all(1)
        points[~valid]=.5
        return self.transform(image),torch.from_numpy(points.astype(np.float32)),torch.from_numpy(valid)

def seed_worker(worker_id):
    seed=torch.initial_seed()%(2**32)
    random.seed(seed); np.random.seed(seed)

datasets={name:FreiHandTiny(ids,name=="train") for name,ids in SPLIT_IDS.items()}
loader_generator=torch.Generator().manual_seed(CFG["SEED"])
loaders={name:DataLoader(dataset,batch_size=CFG["BATCH_SIZE"],shuffle=name=="train",
    num_workers=CFG["NUM_WORKERS"],pin_memory=DEVICE.type=="cuda",
    persistent_workers=False,worker_init_fn=seed_worker,
    generator=loader_generator if name=="train" else torch.Generator().manual_seed(CFG["SEED"]))
    for name,dataset in datasets.items()}
print({name:len(ds) for name,ds in datasets.items()})


## 3. Kiểm tra nhãn, kiểm tra học thuộc, train/resume
Cần xem overlay trước khi tin metric. Sanity check dùng 16 ảnh train không augmentation và một model tạm;
trọng số này được bỏ sau kiểm tra. Sanity check không đạt sẽ dừng để kiểm tra pipeline, không train hàng giờ tiếp.
Mục tiêu sanity dễ hơn mục tiêu tổng quát hóa; đạt sanity không chứng minh validation đạt 90%.


In [ ]:
def heatmap_loss(logits,points,valid):
    size=logits.shape[-1]
    grid=torch.arange(size,device=points.device,dtype=torch.float32)
    yy,xx=torch.meshgrid(grid,grid,indexing="ij")
    centers=points.float()*size-.5
    targets=torch.exp(-((xx[None,None]-centers[:,:,0,None,None]).square()+
                        (yy[None,None]-centers[:,:,1,None,None]).square())/(2*CFG["SIGMA"]**2))
    targets/=targets.sum((-2,-1),keepdim=True).clamp_min(1e-12)
    logs=F.log_softmax(logits.float().flatten(2),dim=-1)
    ce=-(targets.flatten(2)*logs).sum(-1)
    prob=logs.exp().reshape_as(logits)
    ex=(prob*(xx+.5)/size).sum((-2,-1))
    ey=(prob*(yy+.5)/size).sum((-2,-1))
    expected=torch.stack([ex,ey],dim=-1)
    coord=F.smooth_l1_loss(expected,points.float(),reduction="none",beta=.05).sum(-1)
    return ((ce+CFG["COORD_LOSS_WEIGHT"]*coord)*valid).sum()/valid.sum().clamp_min(1)

model=HandPoseAccuracy(pretrained=CFG["PRETRAINED"] and CFG["MODE"]=="train").to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=CFG["LR"],weight_decay=CFG["WEIGHT_DECAY"])
def lr_factor(epoch):
    if epoch<CFG["WARMUP_EPOCHS"]: return (epoch+1)/CFG["WARMUP_EPOCHS"]
    progress=(epoch-CFG["WARMUP_EPOCHS"])/max(1,CFG["EPOCHS"]-CFG["WARMUP_EPOCHS"])
    return float(.01+.99*.5*(1+np.cos(np.pi*min(progress,1))))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lr_factor)
scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
def run_epoch(
    loader: DataLoader,
    training: bool = False,
    phase: str = "Eval",
    epoch: int | None = None,
) -> dict[str, object]:
    model.train(training)
    phase_label = f"{phase} epoch {epoch}" if epoch is not None else phase
    phase_started = time.perf_counter()
    total_batches = len(loader)
    print(f"[{phase_label}] start: {total_batches} batches", flush=True)
    loss_sum = 0.0
    point_count = 0
    error_sum = np.zeros(21, np.float64)
    correct_sum = np.zeros(21, np.float64)
    joint_count = np.zeros(21, np.float64)
    with torch.set_grad_enabled(training):
        for batch_index, (images, points, valid) in enumerate(loader, start=1):
            images = images.to(DEVICE, non_blocking=True)
            points = points.to(DEVICE, non_blocking=True)
            valid = valid.to(DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=training and DEVICE.type == "cuda",
            ):
                logits = model(images)
            loss = heatmap_loss(logits, points, valid)
            if not torch.isfinite(loss):
                raise FloatingPointError("Loss không hữu hạn; kiểm tra dữ liệu/learning rate.")
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
            with torch.no_grad():
                xy = decode_heatmaps(logits.detach())
                error = (xy - points).norm(dim=-1) * IMAGE_SIZE
                count = int(valid.sum())
                loss_sum += float(loss.detach()) * count
                point_count += count
                error_sum += (error * valid).sum(0).cpu().numpy()
                correct_sum += ((error <= 0.05 * IMAGE_SIZE) & valid).sum(0).cpu().numpy()
                joint_count += valid.sum(0).cpu().numpy()
            if (
                batch_index == 1
                or batch_index % CFG["LOG_EVERY_BATCHES"] == 0
                or batch_index == total_batches
            ):
                elapsed = time.perf_counter() - phase_started
                mean_loss = loss_sum / max(point_count, 1)
                print(
                    f"[{phase_label}] batch {batch_index}/{total_batches} "
                    f"loss={mean_loss:.4f} elapsed={elapsed:.0f}s",
                    flush=True,
                )
    result = {
        "loss": loss_sum / max(point_count, 1),
        "epe_px": float(error_sum.sum() / max(joint_count.sum(), 1)),
        "pck_005_crop": float(correct_sum.sum() / max(joint_count.sum(), 1)),
        "per_joint_epe": np.divide(
            error_sum, joint_count, out=np.full(21, np.nan), where=joint_count > 0
        ),
    }
    print(
        f"[{phase_label}] done: loss={result['loss']:.4f} "
        f"EPE={result['epe_px']:.2f}px PCK={result['pck_005_crop']:.3f} "
        f"time={time.perf_counter() - phase_started:.0f}s",
        flush=True,
    )
    return result


def save_training_curves(rows: list[dict[str, float]]) -> None:
    frame = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    panels = [
        ("loss", "Loss", "lower is better"),
        ("epe", "EPE (pixels at 128×128)", "lower is better"),
        ("pck", "PCK@0.05", "higher is better"),
    ]
    for axis, (key, title, subtitle) in zip(axes, panels):
        axis.plot(frame["epoch"], frame[f"train_{key}"], marker="o", label="Train")
        axis.plot(frame["epoch"], frame[f"val_{key}"], marker="o", label="Validation")
        axis.set(title=f"{title}\n{subtitle}", xlabel="Epoch")
        axis.grid(alpha=0.3)
        axis.legend()
    fig.suptitle("HandPose Accuracy training history")
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "training_curves.png", dpi=180, bbox_inches="tight")
    plt.close(fig)




history=[]; best_pck=-1.; best_epe=float("inf"); stale=0; start_epoch=1
checkpoint_path=RUN_DIR/"handpose_accuracy_best.pth"
last_path=RUN_DIR/"handpose_accuracy_last.pth"
if SOURCE_CHECKPOINT is not None:
    saved=torch.load(SOURCE_CHECKPOINT,map_location="cpu",weights_only=True)
    if saved.get("architecture")!="HandPoseAccuracy": raise ValueError("Checkpoint phải thuộc kiến trúc v3.")
    for key in ["SEED","TRAIN_FRACTION","VAL_FRACTION","EXPECTED_BASE_SAMPLES"]:
        if saved["config"][key]!=CFG[key]: raise ValueError(f"Không đổi {key} khi dùng checkpoint")
    model.load_state_dict(saved["model_state"])
    if CFG["MODE"]=="reuse":
        shutil.copy2(SOURCE_CHECKPOINT,checkpoint_path)
    else:
        for key in ["EPOCHS","WARMUP_EPOCHS","LR","BATCH_SIZE","SIGMA","COORD_LOSS_WEIGHT"]:
            if saved["config"][key]!=CFG[key]: raise ValueError(f"Giữ {key} như lúc lưu để resume lịch train")
        if "optimizer" not in saved: raise ValueError("Resume cần file last.pth; best.pth chỉ dùng reuse")
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])
        history=saved["history"]; best_pck=saved["best_pck"];best_epe=saved["best_epe"];stale=saved["stale"]
        start_epoch=saved["epoch"]+1
        random.setstate(saved["python_rng"])
        np.random.set_state((saved["numpy_rng"][0],np.array(saved["numpy_rng"][1],dtype=np.uint32),
                             saved["numpy_rng"][2],saved["numpy_rng"][3],saved["numpy_rng"][4]))
        torch.set_rng_state(saved["torch_rng"])
        loader_generator.set_state(saved["loader_rng"])
        if DEVICE.type=="cuda" and saved.get("cuda_rng"): torch.cuda.set_rng_state_all(saved["cuda_rng"])
        torch.save(saved["best_checkpoint"],checkpoint_path)

# Overlay nhãn gốc: không augmentation.
audit_dataset=FreiHandTiny(SPLIT_IDS["train"][:16],False)
fig,axes=plt.subplots(2,4,figsize=(14,7))
for i,ax in enumerate(axes.flat):
    img,points,valid=audit_dataset[4*i]
    rgb=(img*torch.tensor(STD)[:,None,None]+torch.tensor(MEAN)[:,None,None]).clamp(0,1).permute(1,2,0)
    ax.imshow(rgb); xy=points.numpy()*IMAGE_SIZE
    ax.scatter(xy[valid,0],xy[valid,1],s=12,c="lime")
    for j,(x,y) in enumerate(xy): ax.text(x,y,str(j),fontsize=6,color="red")
    ax.axis("off")
fig.tight_layout();fig.savefig(REPORT_DIR/"label_audit.png",dpi=160);display(fig);plt.close(fig)

if CFG["RUN_OVERFIT_CHECK"] and CFG["MODE"]=="train":
    probe=copy.deepcopy(model)
    probe_opt=torch.optim.AdamW(probe.parameters(),lr=1e-3,weight_decay=0.)
    mini=[audit_dataset[4*i] for i in range(16)]
    ims=torch.stack([x[0] for x in mini]).to(DEVICE)
    pts=torch.stack([x[1] for x in mini]).to(DEVICE)
    val=torch.stack([x[2] for x in mini]).to(DEVICE)
    probe.train()
    for step in range(CFG["OVERFIT_STEPS"]):
        probe_opt.zero_grad(set_to_none=True)
        logits=probe(ims)
        loss=heatmap_loss(logits,pts,val)
        if not torch.isfinite(loss): raise FloatingPointError("Sanity loss non-finite")
        loss.backward();probe_opt.step()
        if (step+1)%50==0: print("Overfit sanity",step+1,float(loss.detach()),flush=True)
    probe.eval()
    with torch.inference_mode():
        error=(decode_heatmaps(probe(ims))-pts).norm(dim=-1)*IMAGE_SIZE
        sanity=float(((error<=6.4)&val).sum()/val.sum())
    write_json(REPORT_DIR/"overfit_sanity.json",{"pck":sanity,"target":.95,"samples":16})
    del probe,probe_opt,ims,pts,val,logits,loss
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    if sanity<.95: raise RuntimeError(f"Sanity chỉ đạt {sanity:.1%}. Xem label_audit.png; kiểm tra dữ liệu hoặc tăng OVERFIT_STEPS trước khi train đầy đủ.")
    print("Overfit sanity passed:",sanity)

if CFG["MODE"]!="reuse":
    for epoch in range(start_epoch,CFG["EPOCHS"]+1):
        started=time.perf_counter()
        train_result=run_epoch(loaders["train"],training=True,phase="Train",epoch=epoch)
        val_result=run_epoch(loaders["val"],phase="Validation",epoch=epoch)
        scheduler.step()
        row={"epoch":epoch,"seconds":time.perf_counter()-started,
             "train_loss":train_result["loss"],"val_loss":val_result["loss"],
             "train_epe":train_result["epe_px"],"val_epe":val_result["epe_px"],
             "train_pck":train_result["pck_005_crop"],"val_pck":val_result["pck_005_crop"]}
        history.append(row)
        improved=row["val_pck"]>best_pck+1e-5 or (abs(row["val_pck"]-best_pck)<=1e-5 and row["val_epe"]<best_epe)
        if improved:
            best_pck=row["val_pck"];best_epe=row["val_epe"];stale=0
            best={"model_state":{k:v.detach().cpu() for k,v in model.state_dict().items()},
                  "architecture":"HandPoseAccuracy","epoch":epoch,"best_val_epe_px":best_epe,
                  "best_val_pck":best_pck,"image_size":IMAGE_SIZE,"heatmap_size":HEATMAP_SIZE,
                  "mean":MEAN,"std":STD,"config":CFG}
            torch.save(best,checkpoint_path)
        else: stale+=1
        nr=np.random.get_state()
        state={"model_state":{k:v.detach().cpu() for k,v in model.state_dict().items()},
               "architecture":"HandPoseAccuracy","epoch":epoch,"config":CFG,
               "optimizer":optimizer.state_dict(),"scheduler":scheduler.state_dict(),"scaler":scaler.state_dict(),
               "history":history,"best_pck":best_pck,"best_epe":best_epe,"stale":stale,
               "best_checkpoint":torch.load(checkpoint_path,map_location="cpu",weights_only=True),
               "python_rng":random.getstate(),"numpy_rng":(nr[0],nr[1].tolist(),nr[2],nr[3],nr[4]),
               "torch_rng":torch.get_rng_state(),"loader_rng":loader_generator.get_state(),
               "cuda_rng":torch.cuda.get_rng_state_all() if DEVICE.type=="cuda" else []}
        temp=last_path.with_suffix(".tmp")
        torch.save(state,temp);temp.replace(last_path)
        del state
        pd.DataFrame(history).to_csv(RUN_DIR/"history.csv",index=False)
        save_training_curves(history)
        print(f"Epoch {epoch}: val PCK={row['val_pck']:.2%}, best={best_pck:.2%}, target={CFG['TARGET_PCK']:.0%}",flush=True)
        if epoch>=CFG["MIN_EPOCHS"] and stale>=CFG["PATIENCE"]:
            print("Early stopping: validation không cải thiện. Không có nghĩa mục tiêu đã đạt.")
            break
else:
    print("Reuse checkpoint v3, bỏ qua training.")


## 4. Đánh giá checkpoint tốt nhất trên test FP32
Checkpoint chọn theo validation PCK; test không tham gia chọn epoch.


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint["model_state"])
model.to(DEVICE).eval()
test_result = run_epoch(loaders["test"], phase="Test")
metrics = {
    "best_epoch": checkpoint["epoch"],
    "val_epe_px": checkpoint["best_val_epe_px"],
    "test_epe_px_128": test_result["epe_px"],
    "test_pck_005_crop": test_result["pck_005_crop"],
    "fingertip_epe_px_128": {
        name: float(test_result["per_joint_epe"][indices[-1]])
        for name, indices in FINGERS.items()
    },
}
(RUN_DIR / "test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("TEST METRICS:", json.dumps(metrics, indent=2), flush=True)

joint_names = ["wrist"] * 21
for finger_name, indices in FINGERS.items():
    for order, joint_index in enumerate(indices, start=1):
        joint_names[joint_index] = f"{finger_name}_{order}"
per_joint_test = pd.DataFrame(
    {
        "joint_index": np.arange(21),
        "joint_name": joint_names,
        "epe_px_128": test_result["per_joint_epe"],
    }
)
per_joint_test.to_csv(REPORT_DIR / "test_per_joint_epe.csv", index=False)
fig, axis = plt.subplots(figsize=(12, 5))
axis.bar(per_joint_test["joint_name"], per_joint_test["epe_px_128"], color="#2878B5")
axis.set(title="FP32 test error by hand joint", ylabel="EPE (pixels at 128×128)", xlabel="Joint")
axis.tick_params(axis="x", rotation=65)
axis.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(REPORT_DIR / "test_per_joint_epe.png", dpi=180, bbox_inches="tight")
plt.close(fig)


def save_prediction_examples(sample_count: int = 8) -> None:
    connections = []
    for indices in FINGERS.values():
        connections.append((0, indices[0]))
        connections.extend(zip(indices[:-1], indices[1:]))
    loader = DataLoader(datasets["test"], batch_size=sample_count, shuffle=False, num_workers=0)
    images, targets, _ = next(iter(loader))
    with torch.no_grad():
        predicted = decode_heatmaps(model(images.to(DEVICE))).cpu()
    mean = torch.tensor(MEAN).view(1, 3, 1, 1)
    std = torch.tensor(STD).view(1, 3, 1, 1)
    shown = (images * std + mean).clamp(0, 1).permute(0, 2, 3, 1).numpy()
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for index, axis in enumerate(axes.flat):
        axis.imshow(shown[index])
        truth = targets[index].numpy() * IMAGE_SIZE
        guess = predicted[index].numpy() * IMAGE_SIZE
        for start, end in connections:
            axis.plot(guess[[start, end], 0], guess[[start, end], 1], color="#00E676", linewidth=1)
        axis.scatter(truth[:, 0], truth[:, 1], s=10, c="#FF1744", label="Ground truth")
        axis.scatter(guess[:, 0], guess[:, 1], s=10, c="#00E676", label="Prediction")
        axis.set_title(f"Sample {index + 1}")
        axis.axis("off")
    axes.flat[0].legend(loc="lower right", fontsize=7)
    fig.suptitle("FP32 predictions: red = ground truth, green = prediction")
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "prediction_examples_fp32.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


save_prediction_examples()
display(FileLink(str(REPORT_DIR / "prediction_examples_fp32.png")))

float_goal = {"target_pck": CFG["TARGET_PCK"], "validation_pck": checkpoint["best_val_pck"],
              "test_pck": metrics["test_pck_005_crop"]}
float_goal["reached"] = float_goal["validation_pck"] >= CFG["TARGET_PCK"] and float_goal["test_pck"] >= CFG["TARGET_PCK"]
write_json(REPORT_DIR/"target_90_status.json",float_goal)
print("FLOAT TARGET:", "REACHED" if float_goal["reached"] else "NOT_REACHED", float_goal)


## 5. Xuất TorchScript/ONNX và kiểm tra tương đương
Cần Internet để cài ONNX Runtime. Không nâng cấp Torch/Torchvision của Kaggle.


In [ ]:
run_checked([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript"], log_name="dependencies.log")
import onnx
import onnxruntime as ort
from onnxruntime.quantization import CalibrationDataReader, CalibrationMethod, QuantFormat, QuantType, quantize_static
from onnxruntime.quantization.shape_inference import quant_pre_process

model_cpu = copy.deepcopy(model).cpu().float().eval()
example = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
traced_path = RUN_DIR / "handpose_accuracy_fp32.pt"
traced = torch.jit.freeze(torch.jit.trace(model_cpu, example, strict=True))
traced.save(str(traced_path))
onnx_path = RUN_DIR / "handpose_accuracy_fp32.onnx"
torch.onnx.export(model_cpu, example, str(onnx_path), input_names=["image"], output_names=["heatmaps"],
                  opset_version=18, dynamic_axes={"image": {0: "batch"}, "heatmaps": {0: "batch"}}, dynamo=False)
onnx.checker.check_model(str(onnx_path))
onnx_preprocessed_path = RUN_DIR / "handpose_accuracy_preprocessed.onnx"
quant_pre_process(input_model=str(onnx_path), output_model_path=str(onnx_preprocessed_path))

def ort_session(path):
    options = ort.SessionOptions()
    options.intra_op_num_threads = CFG["CPU_THREADS"]
    options.inter_op_num_threads = 1
    return ort.InferenceSession(str(path), sess_options=options, providers=["CPUExecutionProvider"])

parity_input = torch.stack([datasets["val"][i][0] for i in [0, 4, 8, 12]])
with torch.inference_mode():
    reference = model_cpu(parity_input).numpy()
parity = {}
for path in [onnx_path, onnx_preprocessed_path]:
    out = ort_session(path).run(["heatmaps"], {"image": parity_input.numpy()})[0]
    parity[path.name] = {"max_abs_diff": float(np.max(np.abs(out-reference)))}
    np.testing.assert_allclose(out, reference, rtol=2e-3, atol=2e-3)
write_json(REPORT_DIR / "onnx_export_parity.json", parity)
print("ONNX export parity passed:", parity)


## 6. Calibration đa dạng, tách khỏi validation/test
Mỗi base ID train lấy ngẫu nhiên một biến thể; không ColorJitter và không lật ảnh. Tensor float32 CHW đã chuẩn hóa được dùng chung cho ONNX và NCNN.


In [ ]:
calibration_dir = RUN_DIR / "calibration_npy"
calibration_dir.mkdir(exist_ok=True)
calibration_list = RUN_DIR / "calibration.txt"
calibration_paths = []
calibration_rows = []
rng = np.random.default_rng(CFG["SEED"] + 100)
cal_ids = rng.choice(SPLIT_IDS["train"], size=min(CFG["CALIBRATION_SAMPLES"], len(SPLIT_IDS["train"])), replace=False)
assert len(cal_ids) > 0
cal_transform = make_transform(False)
for i, base_id in enumerate(tqdm(cal_ids, desc="Calibration tensors")):
    image_path = str(rng.choice(images_by_base[int(base_id)]))
    with Image.open(image_path) as im:
        value = cal_transform(im.convert("RGB")).numpy().astype(np.float32)
    assert value.shape == (3, IMAGE_SIZE, IMAGE_SIZE) and np.isfinite(value).all()
    path = calibration_dir / f"{i:05d}.npy"
    np.save(path, value)
    calibration_paths.append(str(path))
    calibration_rows.append({"base_id": int(base_id), "source": image_path, "tensor": str(path)})
calibration_list.write_text("\n".join(calibration_paths)+"\n", encoding="utf-8")
pd.DataFrame(calibration_rows).to_csv(REPORT_DIR / "calibration_manifest.csv", index=False)
print("Số bàn tay độc lập dùng calibration:", len(cal_ids))

class CalibrationReader(CalibrationDataReader):
    def __init__(self, paths):
        self.paths = paths
        self.rewind()
    def get_next(self):
        if self.index >= len(self.paths): return None
        x = np.load(self.paths[self.index], allow_pickle=False)[None].astype(np.float32)
        self.index += 1
        return {"image": x}
    def rewind(self): self.index = 0


## 7. Hàm đánh giá và cổng chất lượng
EPE tính theo pixel ảnh 128×128; PCK dùng khoảng cách ≤ 6,4 px. Cổng INT8: EPE tăng ≤ 1 px **và** PCK giảm ≤ 0,02 so với FP32 cùng backend.

V3 thêm điều kiện tuyệt đối PCK ≥ 90% ngoài hai giới hạn suy giảm.


In [ ]:
class MetricAccumulator:
    def __init__(self):
        self.error = np.zeros(21, np.float64)
        self.correct = np.zeros(21, np.float64)
        self.count = np.zeros(21, np.float64)
    def add(self, predicted, points, valid):
        predicted = np.asarray(predicted)
        points, valid = np.asarray(points), np.asarray(valid, dtype=bool)
        if not np.isfinite(predicted).all(): raise FloatingPointError("Non-finite predictions")
        error = np.linalg.norm(predicted - points, axis=-1) * IMAGE_SIZE
        self.error += np.where(valid, error, 0).sum(0)
        self.correct += ((error <= 0.05 * IMAGE_SIZE) & valid).sum(0)
        self.count += valid.sum(0)
    def result(self):
        if self.count.sum() == 0: raise ValueError("No valid keypoints")
        return {"epe_px": float(self.error.sum()/self.count.sum()),
                "pck_005_crop": float(self.correct.sum()/self.count.sum()),
                "per_joint_epe": np.divide(self.error, self.count, out=np.full(21, np.nan), where=self.count>0).tolist(),
                "valid_keypoints": int(self.count.sum())}

def evaluate_onnx(path, dataset):
    session = ort_session(path)
    acc = MetricAccumulator()
    loader = DataLoader(dataset, batch_size=CFG["EVAL_BATCH_SIZE"], shuffle=False, num_workers=0)
    for images, points, valid in tqdm(loader, desc=Path(path).stem):
        logits = session.run(["heatmaps"], {"image": images.numpy()})[0]
        if not np.isfinite(logits).all(): raise FloatingPointError("Non-finite ONNX logits")
        xy = decode_heatmaps(torch.from_numpy(logits)).numpy()
        acc.add(xy, points.numpy(), valid.numpy())
    return acc.result()

def quality_gate(base, quant):
    de = quant["epe_px"] - base["epe_px"]
    dp = quant["pck_005_crop"] - base["pck_005_crop"]
    passed = bool(np.isfinite([de, dp]).all() and de <= CFG["MAX_EPE_DELTA"] and dp >= -CFG["MAX_PCK_DROP"])
    target_ok = bool(quant["pck_005_crop"] >= CFG["TARGET_PCK"])
    return {"passed": passed and target_ok, "relative_gate_passed": passed,
            "absolute_target_passed": target_ok, "target_pck": CFG["TARGET_PCK"],
            "delta_epe_px": float(de), "delta_pck": float(dp)}

def choose_candidate(rows):
    available = [r for r in rows if "validation" in r]
    if not available: return None
    passed = [r for r in available if r["validation_gate"]["passed"]]
    return min(passed or available, key=lambda r: (r["validation"]["epe_px"], -r["validation"]["pck_005_crop"]))


## 8. Thử ONNX PTQ trên validation, chốt rồi đánh giá test
Thử ba phương pháp trên cùng tập calibration. Nếu không cấu hình nào đạt, thử Entropy với convolution
đầu và cuối giữ FP32. Bản đó là **mixed precision**, không phải mọi convolution đều INT8.
Không đổi ngưỡng để ép đạt và không dùng test để chọn ứng viên. Các lần thử có thể mất nhiều phút trên CPU.

Model lớn hơn nên mặc định calibration dùng 256 base ID độc lập để hạn chế RAM của Entropy. Có thể tăng khi có đủ RAM.


In [ ]:
onnx_report = {"status": "running", "candidates": []}
onnx_selected = None
float_onnx_test = None
try:
    float_onnx_val = evaluate_onnx(onnx_preprocessed_path, datasets["val"])
    onnx_report["float_validation"] = float_onnx_val
    graph = onnx.load(str(onnx_preprocessed_path))
    convs = [node.name for node in graph.graph.node if node.op_type == "Conv"]
    if not convs or any(not name for name in convs):
        raise ValueError("Expected named Conv nodes in exported model")
    experiments = [
        ("s8_minmax", CalibrationMethod.MinMax, []),
        ("s8_entropy", CalibrationMethod.Entropy, []),
        ("s8_percentile", CalibrationMethod.Percentile, []),
        ("mixed_entropy", CalibrationMethod.Entropy, list(dict.fromkeys([convs[0], convs[-1]]))),
    ]
    for name, method, excluded in experiments:
        if excluded and any(r.get("validation_gate", {}).get("passed", False) for r in onnx_report["candidates"]):
            break
        path = RUN_DIR / f"{name}.onnx"
        record = {"name": name, "path": str(path), "calibration": method.name,
                  "precision": "mixed_fp32_int8" if excluded else "int8_convolutions",
                  "excluded_nodes": excluded}
        try:
            extra = {"CalibPercentile": 99.99} if method == CalibrationMethod.Percentile else {}
            quantize_static(model_input=str(onnx_preprocessed_path), model_output=str(path),
                            calibration_data_reader=CalibrationReader(calibration_paths),
                            quant_format=QuantFormat.QDQ, activation_type=QuantType.QInt8,
                            weight_type=QuantType.QInt8, per_channel=True,
                            op_types_to_quantize=["Conv"], nodes_to_exclude=excluded,
                            calibrate_method=method, extra_options=extra)
            onnx.checker.check_model(str(path))
            record["validation"] = evaluate_onnx(path, datasets["val"])
            record["validation_gate"] = quality_gate(float_onnx_val, record["validation"])
            record["size_kib"] = path.stat().st_size / 1024
            print(name, record["validation_gate"])
        except Exception as exc:
            record["error"] = repr(exc)
            print("Candidate failed:", name, exc)
        onnx_report["candidates"].append(record)
        write_json(REPORT_DIR / "onnx_quantization.json", onnx_report)
    onnx_selected = choose_candidate(onnx_report["candidates"])
    float_onnx_test = evaluate_onnx(onnx_preprocessed_path, datasets["test"])
    onnx_report["float_test"] = float_onnx_test
    if onnx_selected:
        selected_test = evaluate_onnx(Path(onnx_selected["path"]), datasets["test"])
        onnx_report["selected"] = onnx_selected
        onnx_report["selected_test"] = selected_test
        onnx_report["test_gate"] = quality_gate(float_onnx_test, selected_test)
        onnx_report["passed"] = onnx_selected["validation_gate"]["passed"] and onnx_report["test_gate"]["passed"]
        onnx_report["status"] = "passed" if onnx_report["passed"] else "failed_quality_gate"
    else:
        onnx_report["status"] = "no_valid_candidate"
except Exception as exc:
    onnx_report.update(status="error", error=repr(exc), traceback=traceback.format_exc())
    print(onnx_report["traceback"])
write_json(REPORT_DIR / "onnx_quantization.json", onnx_report)
print("ONNX status:", onnx_report["status"])


## 9. NCNN: build công cụ và runner cùng một phiên bản
NCNN được chuyển trực tiếp từ TorchScript FP32, **không** chuyển từ ONNX INT8.
PNNX được ép `fp16=0`. Calibration `.npy` có thứ tự CHW; `shape` của `ncnn2table` là WHC.
Runner C++ bên dưới được liên kết cùng bản NCNN dùng để lượng tử hóa, không phụ thuộc wheel Python NCNN.
Runner đọc tensor float32 CHW đã chuẩn hóa từ stdin, xuất 21 cặp xy float32 chuẩn hóa ra stdout.
Binary build trên Kaggle là cho máy Kaggle; khi chạy BBB cần biên dịch lại source cho ARM.


In [ ]:
NCNN_RUNNER_SOURCE = '#include <net.h>\n#include <algorithm>\n#include <cmath>\n#include <cstring>\n#include <iostream>\n#include <vector>\nint main(int argc, char** argv) {\n    if (argc != 3) { std::cerr << "usage: pose_runner model.param model.bin\\n"; return 1; }\n    ncnn::Net net;\n    net.opt.num_threads = 1;\n    net.opt.use_vulkan_compute = false;\n    net.opt.use_fp16_packed = false;\n    net.opt.use_fp16_storage = false;\n    net.opt.use_fp16_arithmetic = false;\n    net.opt.use_packing_layout = false;\n    net.opt.use_int8_inference = true;\n    if (net.load_param(argv[1]) || net.load_model(argv[2])) return 2;\n    const auto inputs = net.input_names();\n    const auto outputs = net.output_names();\n    if (inputs.size()!=1 || outputs.size()!=1) return 3;\n    std::vector<float> buffer(3*128*128);\n    while (std::cin.read(reinterpret_cast<char*>(buffer.data()), buffer.size()*sizeof(float))) {\n        ncnn::Mat in(128,128,3);\n        for (int c=0;c<3;c++) std::memcpy(in.channel(c), buffer.data()+c*128*128, 128*128*sizeof(float));\n        ncnn::Extractor ex = net.create_extractor();\n        if (ex.input(inputs[0], in)) return 4;\n        ncnn::Mat out;\n        if (ex.extract(outputs[0], out)) return 5;\n        if (out.w!=64 || out.h!=64 || out.c!=21 || out.elempack!=1 || out.elemsize!=sizeof(float)) return 6;\n        float xy[42];\n        for (int j=0;j<21;j++) {\n            const float* h = out.channel(j);\n            int best=0;\n            for (int k=0;k<4096;k++) {\n                if (!std::isfinite(h[k])) return 7;\n                if (h[k]>h[best]) best=k;\n            }\n            xy[2*j] = (best%64+0.5f)/64;\n            xy[2*j+1] = (best/64+0.5f)/64;\n        }\n        std::cout.write(reinterpret_cast<char*>(xy), sizeof(xy));\n        std::cout.flush();\n        if (!std::cout) return 8;\n    }\n    return std::cin.gcount()==0 ? 0 : 9;\n}\n'
NCNN_CMAKE = 'cmake_minimum_required(VERSION 3.10)\nproject(pose_runner LANGUAGES CXX)\nset(CMAKE_CXX_STANDARD 11)\nfind_package(ncnn REQUIRED)\nadd_executable(pose_runner pose_runner.cpp)\ntarget_link_libraries(pose_runner PRIVATE ncnn)\n'
runner_source = RUN_DIR / "ncnn_runner_source"
runner_source.mkdir(exist_ok=True)
(runner_source / "pose_runner.cpp").write_text(NCNN_RUNNER_SOURCE)
(runner_source / "CMakeLists.txt").write_text(NCNN_CMAKE)
ncnn_report = {"status": "not_requested", "candidates": []}
ncnn_ready = False
if CFG["BUILD_NCNN"]:
    try:
        for command in ["git", "cmake", "c++"]:
            if not shutil.which(command): raise RuntimeError(f"Thiếu công cụ: {command}")
        run_checked([sys.executable, "-m", "pip", "install", "-q", "pnnx"], log_name="ncnn_build.log")
        tools_root = Path(CFG["OUTPUT_BASE"]) / "tiny_handpose_ncnn_tools_v2"
        repo, build, install = tools_root/"ncnn", tools_root/"build", tools_root/"install"
        tools_root.mkdir(exist_ok=True, parents=True)
        if not repo.exists():
            run_checked(["git", "clone", "--depth=1", "https://github.com/Tencent/ncnn.git", repo], log_name="ncnn_build.log")
        if CFG["NCNN_REF"] != "master":
            run_checked(["git", "fetch", "--depth=1", "origin", CFG["NCNN_REF"]], cwd=repo, log_name="ncnn_build.log")
            run_checked(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo, log_name="ncnn_build.log")
        ncnn_report["source_commit"] = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
        run_checked(["cmake", "-S", repo, "-B", build, "-DCMAKE_BUILD_TYPE=Release",
                     f"-DCMAKE_INSTALL_PREFIX={install}", "-DNCNN_BUILD_TOOLS=ON",
                     "-DNCNN_BUILD_EXAMPLES=OFF", "-DNCNN_BUILD_BENCHMARK=OFF", "-DNCNN_BUILD_TESTS=OFF",
                     "-DNCNN_INT8=ON", "-DNCNN_VULKAN=OFF"], log_name="ncnn_build.log")
        run_checked(["cmake", "--build", build, f"-j{CFG['BUILD_JOBS']}"], log_name="ncnn_build.log")
        run_checked(["cmake", "--install", build], log_name="ncnn_build.log")
        package_configs = list(install.rglob("ncnnConfig.cmake"))
        if len(package_configs) != 1: raise RuntimeError("Không tìm được ncnnConfig.cmake duy nhất")
        runner_build = tools_root / "runner_build"
        run_checked(["cmake", "-S", runner_source, "-B", runner_build,
                     f"-Dncnn_DIR={package_configs[0].parent}", "-DCMAKE_BUILD_TYPE=Release"], log_name="ncnn_build.log")
        run_checked(["cmake", "--build", runner_build, f"-j{CFG['BUILD_JOBS']}"], log_name="ncnn_build.log")
        runner_binary = runner_build / "pose_runner"
        fp32_param = RUN_DIR / "handpose_accuracy_fp32.ncnn.param"
        fp32_bin = RUN_DIR / "handpose_accuracy_fp32.ncnn.bin"
        pnnx_exe = shutil.which("pnnx")
        if not pnnx_exe: raise RuntimeError("pnnx executable not on PATH")
        run_checked([pnnx_exe, traced_path, f"inputshape=[1,3,{IMAGE_SIZE},{IMAGE_SIZE}]", "fp16=0",
                     f"ncnnparam={fp32_param}", f"ncnnbin={fp32_bin}"], cwd=RUN_DIR, log_name="pnnx.log")
        for p in [fp32_param, fp32_bin, runner_binary]:
            if not p.is_file() or not p.stat().st_size: raise FileNotFoundError(p)
        ncnn2table = build / "tools/quantize/ncnn2table"
        ncnn2int8 = build / "tools/quantize/ncnn2int8"
        ncnn_report["status"] = "built"
        ncnn_ready = True
    except Exception as exc:
        ncnn_report.update(status="build_error", error=repr(exc), traceback=traceback.format_exc())
        print(ncnn_report["traceback"])
write_json(REPORT_DIR / "ncnn_quantization.json", ncnn_report)
print("NCNN:", ncnn_report["status"])


## 10. Đánh giá NCNN và lượng tử hóa
Đầu tiên kiểm tra NCNN FP32 so với PyTorch trên validation. Nếu sai khác lớn, dừng riêng nhánh NCNN để tránh lượng tử hóa một graph chuyển đổi sai. Tiếp theo chọn KL/ACIQ trên validation và kiểm tra bản đã chọn trên test.


In [ ]:
def evaluate_ncnn(param, weights, dataset):
    acc = MetricAccumulator()
    log_path = REPORT_DIR / (Path(param).stem + "_runner.log")
    with log_path.open("wb") as err:
        process = subprocess.Popen([str(runner_binary), str(param), str(weights)],
                                   stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=err)
        try:
            for i in tqdm(range(len(dataset)), desc=Path(param).stem):
                image, points, valid = dataset[i]
                process.stdin.write(np.ascontiguousarray(image.numpy(), dtype=np.float32).tobytes())
                process.stdin.flush()
                payload = bytearray()
                while len(payload) < 42*4:
                    piece = process.stdout.read(42*4-len(payload))
                    if not piece: raise RuntimeError(f"NCNN runner stopped; xem {log_path}")
                    payload.extend(piece)
                xy = np.frombuffer(payload, dtype=np.float32).reshape(1,21,2)
                acc.add(xy, points.numpy()[None], valid.numpy()[None])
            process.stdin.close()
            status = process.wait(timeout=30)
            if status: raise RuntimeError(f"NCNN runner exit {status}; xem {log_path}")
        finally:
            if process.poll() is None:
                process.kill()
                process.wait()
            if not process.stdin.closed: process.stdin.close()
            process.stdout.close()
    return acc.result()

ncnn_selected = None
if ncnn_ready:
    try:
        # Smoke check trước khi đánh giá toàn bộ validation.
        smoke = Subset(datasets["val"], list(range(64)))
        smoke_acc = MetricAccumulator()
        for images, points, valid in DataLoader(smoke, batch_size=32):
            with torch.inference_mode():
                xy = decode_heatmaps(model_cpu(images)).numpy()
            smoke_acc.add(xy, points.numpy(), valid.numpy())
        smoke_torch = smoke_acc.result()
        smoke_ncnn = evaluate_ncnn(fp32_param, fp32_bin, smoke)
        ncnn_report["conversion_smoke"] = {"pytorch": smoke_torch, "ncnn": smoke_ncnn}
        if abs(smoke_ncnn["epe_px"]-smoke_torch["epe_px"]) > 0.15 or abs(smoke_ncnn["pck_005_crop"]-smoke_torch["pck_005_crop"]) > 0.01:
            raise RuntimeError("NCNN FP32 lệch PyTorch trên smoke set; xem pnnx.log")
        ncnn_float_val = evaluate_ncnn(fp32_param, fp32_bin, datasets["val"])
        ncnn_report["float_validation"] = ncnn_float_val
        if "float_validation" in onnx_report:
            reference_val = onnx_report["float_validation"]
        else:
            model.to(DEVICE)
            reference_val = run_epoch(loaders["val"], phase="FP32 validation")
        conversion_ok = (abs(ncnn_float_val["epe_px"] - reference_val["epe_px"]) <= 0.15 and
                         abs(ncnn_float_val["pck_005_crop"] - reference_val["pck_005_crop"]) <= 0.01)
        ncnn_report["conversion_validation_passed"] = conversion_ok
        if not conversion_ok: raise RuntimeError("NCNN FP32 không tương đương FP32 reference trên validation")
        for method in CFG["NCNN_METHODS"]:
            table = RUN_DIR / f"ncnn_{method}.table"
            param = RUN_DIR / f"ncnn_int8_{method}.param"
            weights = RUN_DIR / f"ncnn_int8_{method}.bin"
            record = {"name": method, "param": str(param), "bin": str(weights)}
            try:
                run_checked([ncnn2table, fp32_param, fp32_bin, calibration_list, table,
                             f"shape=[{IMAGE_SIZE},{IMAGE_SIZE},3]", f"thread={CFG['CPU_THREADS']}",
                             f"method={method}", "type=1"], log_name=f"ncnn_{method}.log")
                run_checked([ncnn2int8, fp32_param, fp32_bin, param, weights, table], log_name=f"ncnn_{method}.log")
                for p in [param, weights]:
                    if not p.is_file() or not p.stat().st_size: raise FileNotFoundError(p)
                # Param key 8 là int8_scale_term cho các convolution NCNN.
                quantized_layers = []
                for line in param.read_text().splitlines()[2:]:
                    fields = line.split()
                    if fields and fields[0] in {"Convolution", "ConvolutionDepthWise"}:
                        if any(f.startswith("8=") and f != "8=0" for f in fields): quantized_layers.append(fields[1])
                if not quantized_layers: raise RuntimeError("Không có convolution INT8 trong NCNN param")
                record["int8_convolution_count"] = len(quantized_layers)
                record["validation"] = evaluate_ncnn(param, weights, datasets["val"])
                record["validation_gate"] = quality_gate(ncnn_float_val, record["validation"])
                print(method, record["validation_gate"])
            except Exception as exc:
                record["error"] = repr(exc)
                print("NCNN candidate failed:", method, exc)
            ncnn_report["candidates"].append(record)
            write_json(REPORT_DIR / "ncnn_quantization.json", ncnn_report)
        ncnn_selected = choose_candidate(ncnn_report["candidates"])
        ncnn_float_test = evaluate_ncnn(fp32_param, fp32_bin, datasets["test"])
        ncnn_report["float_test"] = ncnn_float_test
        if ncnn_selected:
            qtest = evaluate_ncnn(Path(ncnn_selected["param"]), Path(ncnn_selected["bin"]), datasets["test"])
            ncnn_report["selected"] = ncnn_selected
            ncnn_report["selected_test"] = qtest
            ncnn_report["test_gate"] = quality_gate(ncnn_float_test, qtest)
            ncnn_report["passed"] = ncnn_selected["validation_gate"]["passed"] and ncnn_report["test_gate"]["passed"]
            ncnn_report["status"] = "passed" if ncnn_report["passed"] else "failed_quality_gate"
        else:
            ncnn_report["status"] = "no_valid_candidate"
    except Exception as exc:
        ncnn_report.update(status="evaluation_error", error=repr(exc), traceback=traceback.format_exc())
        print(ncnn_report["traceback"])
write_json(REPORT_DIR / "ncnn_quantization.json", ncnn_report)
print("NCNN final status:", ncnn_report["status"])


## 11a. Gói triển khai BBB và benchmark
Tạo source C++, tensor smoke test và hướng dẫn build. Kết quả FPS/RAM phải đo trên BBB; không suy ra từ Kaggle.


In [ ]:
BBB_BENCHMARK_SOURCE = '#include <net.h>\n#include <algorithm>\n#include <chrono>\n#include <cstring>\n#include <fstream>\n#include <iostream>\n#include <numeric>\n#include <sys/resource.h>\n#include <vector>\nint main(int argc,char** argv){\n if(argc!=4){std::cerr<<"usage: benchmark_bbb model.param model.bin input_chw.f32\\n";return 1;}\n ncnn::Net net;\n net.opt.num_threads=1;net.opt.use_vulkan_compute=false;\n net.opt.use_fp16_packed=false;net.opt.use_fp16_storage=false;net.opt.use_fp16_arithmetic=false;\n net.opt.use_int8_inference=true;\n if(net.load_param(argv[1])||net.load_model(argv[2]))return 2;\n auto names=net.input_names();auto outs=net.output_names();\n if(names.size()!=1||outs.size()!=1)return 3;\n std::vector<float> pixels(3*128*128);\n std::ifstream file(argv[3],std::ios::binary);\n if(!file.read(reinterpret_cast<char*>(pixels.data()),pixels.size()*sizeof(float)))return 4;\n std::vector<double> times;\n for(int i=0;i<110;i++){\n  ncnn::Mat input(128,128,3);\n  for(int c=0;c<3;c++)std::memcpy(input.channel(c),pixels.data()+c*128*128,128*128*sizeof(float));\n  auto start=std::chrono::steady_clock::now();\n  ncnn::Extractor ex=net.create_extractor();\n  if(ex.input(names[0],input))return 5;\n  ncnn::Mat out;\n  if(ex.extract(outs[0],out))return 6;\n  auto stop=std::chrono::steady_clock::now();\n  if(i>=10)times.push_back(std::chrono::duration<double,std::milli>(stop-start).count());\n }\n std::sort(times.begin(),times.end());\n double mean=std::accumulate(times.begin(),times.end(),0.)/times.size();\n rusage usage;getrusage(RUSAGE_SELF,&usage);\n std::cout<<"{\\"mean_ms\\":"<<mean<<",\\"median_ms\\":"<<times[times.size()/2]\n <<",\\"p95_ms\\":"<<times[94]<<",\\"inference_fps\\":"<<1000./mean\n <<",\\"peak_rss_kib\\":"<<usage.ru_maxrss<<",\\"threads\\":1,\\"samples\\":100}\\n";\n}\n'
BBB_README = '# Chạy trên BeagleBone Black (BBB)\n\nChỉ model trong approved_models được xác nhận đạt cổng accuracy của notebook.\nChưa có model trong đó nghĩa là accuracy chưa đạt, không được tự xem là đã hoàn thành.\nKhông cần Python/PyTorch/ONNX Runtime trên BBB: C++ + NCNN CPU, 1 thread, Vulkan OFF.\n\n## Build native trên BBB chạy Linux hard-float ARMv7\nCần git, cmake, g++, make. Clone NCNN theo đúng source_commit trong deployment_metadata.json.\nỞ thư mục artifact đã giải nén:\n\n    git clone https://github.com/Tencent/ncnn.git ncnn\n    git -C ncnn checkout <source_commit-trong-bao-cao>\n    cmake -S ncnn -B ncnn-build -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX="$PWD/ncnn-install" -DNCNN_VULKAN=OFF -DNCNN_INT8=ON -DNCNN_BUILD_TOOLS=OFF -DNCNN_BUILD_EXAMPLES=OFF -DNCNN_BUILD_BENCHMARK=OFF\n    cmake --build ncnn-build -j1\n    cmake --install ncnn-build\n    cmake -S ncnn_runner_source -B runner-build -DCMAKE_BUILD_TYPE=Release -DCMAKE_PREFIX_PATH="$PWD/ncnn-install"\n    cmake --build runner-build -j1\n\nCó thể cross-compile trên Linux laptop bằng toolchain arm-linux-gnueabihf tương thích sysroot của BBB.\nDùng cùng toolchain cho NCNN và runner. Không copy binary x86 từ Kaggle sang BBB.\n\n## Chạy smoke test và benchmark\n\n    ./runner-build/pose_runner approved_models/handpose_accuracy_int8.param approved_models/handpose_accuracy_int8.bin < bbb_smoke_input_chw.f32 > bbb_output_xy.f32\n    ./runner-build/benchmark_bbb approved_models/handpose_accuracy_int8.param approved_models/handpose_accuracy_int8.bin bbb_smoke_input_chw.f32 > bbb_benchmark.json\n\nPose output là 42 float32 native-endian (21 cặp xy chuẩn hóa 0..1). File bbb_reference_xy.json\nchứa kết quả của NCNN host trên cùng tensor, nếu nhánh NCNN đã đánh giá thành công.\nNếu model chưa đạt: có thể benchmark ứng viên gốc bằng cách thay đường dẫn, nhưng vẫn là bản thử nghiệm.\n\nInput tensor đã normalize; không normalize lần hai. Camera cần crop bàn tay, RGB, resize PIL-bilinear-equivalent\n128x128, chia 255, rồi (x-mean)/std và đổi HWC→CHW. Mean/std nằm trong metadata.\nĐổi resize backend có thể gây sai khác; so khớp tensor trước khi so khớp inference.\n\nBenchmark chỉ đo inference cho crop đã chuẩn bị, không gồm camera, detector, resize, vẽ hay truyền dữ liệu.\nPeak RSS là bộ nhớ toàn process trên Linux. Đây chưa phải kết quả đo có sẵn.\nCần người dùng xác định FPS tối thiểu; notebook không tự đặt hoặc công nhận tiêu chí realtime.\n'
(runner_source/"benchmark_bbb.cpp").write_text(BBB_BENCHMARK_SOURCE)
cmake_text=NCNN_CMAKE+"\nadd_executable(benchmark_bbb benchmark_bbb.cpp)\ntarget_link_libraries(benchmark_bbb PRIVATE ncnn)\n"
(runner_source/"CMakeLists.txt").write_text(cmake_text)
(RUN_DIR/"BBB_DEPLOY.md").write_text(BBB_README,encoding="utf-8")
smoke_image,_,_=datasets["val"][0]
smoke_image.numpy().astype(np.float32).tofile(RUN_DIR/"bbb_smoke_input_chw.f32")
if ncnn_report.get("selected"):
    selected=ncnn_report["selected"]
    attempt=subprocess.run([str(runner_binary),selected["param"],selected["bin"]],
        input=smoke_image.numpy().astype(np.float32).tobytes(),stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=60)
    if attempt.returncode==0 and len(attempt.stdout)==42*4:
        write_json(RUN_DIR/"bbb_reference_xy.json",{
            "model":selected["name"], "accuracy_gate_passed":ncnn_report.get("passed",False),
            "xy":np.frombuffer(attempt.stdout,dtype=np.float32).reshape(21,2).tolist()})
    else:
        print("BBB reference generation failed:",attempt.returncode,attempt.stderr.decode(errors="replace")[-500:])


## 11. Báo cáo, model đạt ngưỡng và ZIP
Thư mục `approved_models` chỉ chứa model INT8 đã qua cả validation và test. Model chưa đạt vẫn nằm trong thư mục gốc để chẩn đoán. ZIP không chứa calibration tensors hay build cache, nên nhỏ hơn nhiều.


In [ ]:
rows = [{"backend": "PyTorch FP32", "epe_px": metrics["test_epe_px_128"], "pck": metrics["test_pck_005_crop"], "status": "baseline"}]
for label, report in [("ONNX", onnx_report), ("NCNN", ncnn_report)]:
    if "float_test" in report:
        m = report["float_test"]
        rows.append({"backend": label+" FP32", "epe_px": m["epe_px"], "pck": m["pck_005_crop"], "status": "baseline"})
    if "selected_test" in report:
        m = report["selected_test"]
        selected_label = report.get("selected", {}).get("precision", "INT8")
        rows.append({"backend": label+" "+selected_label, "epe_px": m["epe_px"], "pck": m["pck_005_crop"], "status": report["status"]})
comparison = pd.DataFrame(rows)
comparison.to_csv(REPORT_DIR / "test_comparison.csv", index=False)
display(comparison)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(comparison.backend, comparison.epe_px)
axes[0].set(title="Test EPE — lower is better", ylabel="Pixels at 128×128")
axes[1].bar(comparison.backend, comparison.pck)
axes[1].set(title="Test PCK@0.05 — higher is better", ylabel="Ratio", ylim=(0,1))
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(REPORT_DIR / "test_comparison.png", dpi=160)
display(fig)
plt.close(fig)

approved = RUN_DIR / "approved_models"
approved.mkdir(exist_ok=True)
# Xóa bản approved của lần chạy lại cell trước; ứng viên gốc vẫn được giữ.
for p in approved.iterdir():
    if p.is_file(): p.unlink()
if onnx_report.get("passed") and onnx_report.get("status") == "passed":
    precision = onnx_report["selected"]["precision"]
    shutil.copy2(onnx_report["selected"]["path"], approved / f"tiny_handpose_{precision}.onnx")
if ncnn_report.get("passed") and ncnn_report.get("status") == "passed":
    shutil.copy2(ncnn_report["selected"]["param"], approved / "handpose_accuracy_int8.param")
    shutil.copy2(ncnn_report["selected"]["bin"], approved / "handpose_accuracy_int8.bin")

versions = {"python": platform.python_version(), "torch": str(torch.__version__)}
for package in ["torchvision", "numpy", "onnx", "onnxruntime", "pnnx"]:
    try: versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: pass
write_json(RUN_DIR / "versions.json", versions)
metadata = {
    "task": "single cropped hand, 21 2D keypoints", "architecture": "HandPoseAccuracy",
    "input": [1,3,128,128], "output": [1,21,64,64],
    "preprocess": {"color": "RGB", "resize": "PIL bilinear 128x128", "scale": "divide by 255", "mean": MEAN, "std": STD},
    "decode": "argmax per heatmap, (index+0.5)/64; no softmax",
    "calibration": {"split": "train", "distinct_base_ids": len(cal_ids), "augmentation": False},
    "source_checkpoint": str(SOURCE_CHECKPOINT) if SOURCE_CHECKPOINT else None,
    "thresholds": {"max_epe_increase_px": CFG["MAX_EPE_DELTA"], "max_pck_drop": CFG["MAX_PCK_DROP"]},
    "annotations_sha256": {name: hashlib.sha256((ROOT/name).read_bytes()).hexdigest() for name in ["training_K.json", "training_xyz.json"]},
    "target_pck": CFG["TARGET_PCK"], "float_target": float_goal, "pytorch_test": metrics, "onnx": onnx_report, "ncnn": ncnn_report,
    "bbb_hardware_benchmarked": False,
}
write_json(RUN_DIR / "deployment_metadata.json", metadata)
readme = f"""HandPose Accuracy artifacts
ONNX status: {onnx_report['status']}
NCNN status: {ncnn_report['status']}

approved_models/ contains only quantized candidates with PCK >= 90% on validation AND test, also passing relative quality gates.
Root-level quantized models are diagnostic candidates; check report before deployment.
Passing a relative quantization gate does not certify absolute accuracy or speed on BBB.
If no candidate passes: inspect per-joint errors, calibration distributions and try QAT.
This notebook performs PTQ; it does not silently retrain with QAT.

Input: one RGB hand crop; resize PIL bilinear to 128x128; /255; normalize by
mean={MEAN}, std={STD}; planar CHW float32. Do not normalize twice.
Output: 21 heatmaps 64x64. Decode argmax with normalized center (index+0.5)/64.
To map to original image, multiply normalized xy by crop width/height, add crop origin.
NCNN input/output blob names come from the actual .param / net.input_names/output_names.

ncnn_runner_source/ contains a portable C++ tensor runner and CMakeLists.txt.
Build NCNN for the target architecture first, then build this runner with
cmake -S ncnn_runner_source -B runner_build -Dncnn_DIR=/path/to/ncnn/lib/cmake/ncnn
cmake --build runner_build -j2
Runner stdin: consecutive native-endian float32 CHW tensors (3*128*128 floats each).
Runner stdout: 42 native-endian float32 normalized xy values per image.
No image decoder, camera loop or hand detector is included in the tensor runner.
Do not copy a Kaggle x86 executable to BBB; compile it for ARM.
NCNN source commit: {ncnn_report.get('source_commit', 'not built')}
Measure latency/FPS on BBB itself; host evaluation does not establish target speed.

Calibration tensors and build cache are excluded from ZIP; calibration_manifest.csv
and split_ids.npz allow regeneration with the attached dataset and config.
"""
(RUN_DIR / "README.txt").write_text(readme, encoding="utf-8")
archive_path = RUN_DIR.with_name(RUN_DIR.name + "_artifacts.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUN_DIR.rglob("*")):
        if p.is_file() and "calibration_npy" not in p.relative_to(RUN_DIR).parts:
            z.write(p, p.relative_to(RUN_DIR))
print("FLOAT 90%:", "REACHED" if float_goal["reached"] else "NOT_REACHED")
print("ONNX:", onnx_report["status"], "| NCNN:", ncnn_report["status"])
if ncnn_report.get("status") != "passed":
    print("CHƯA có NCNN INT8 được xác nhận đạt ngưỡng. Xem report/ncnn_quantization.json.")
print("Artifacts:", archive_path)
display(FileLink(str(archive_path)))


## Khi nào được kết luận đã đạt yêu cầu?
1. `report/target_90_status.json`: FLOAT validation và test đều PCK ≥ 0,90.
2. `report/ncnn_quantization.json`: `status="passed"`, tức NCNN INT8 validation/test đều ≥0,90 và không giảm quá giới hạn.
3. Benchmark thực tế BBB về FPS, RAM, độ trễ. Notebook chưa chạy phần cứng BBB, nên không tự công nhận yêu cầu tốc độ.

Nếu FLOAT chưa đạt, xem label audit, overfit sanity, curves và lỗi từng khớp. Nếu train cao nhưng validation thấp,
ưu tiên generalization; nếu cả hai thấp, cần điều chỉnh capacity/optimization và dữ liệu. Không lặp lại lựa chọn model trên test.
Nếu FLOAT đạt nhưng INT8 chưa đạt, cần nghiên cứu calibration/QAT; nếu model quá chậm trên BBB, cần distillation.

**Resume:** tải `handpose_accuracy_last.pth` (chứa cả best checkpoint), giữ tổng EPOCHS/lịch LR và split như cũ,
đặt MODE="resume" và CHECKPOINT. Checkpoint lưu sau mỗi epoch; phần epoch đang chạy khi ngắt sẽ phải chạy lại.
Không dùng checkpoint 70k tham số của notebook cũ cho kiến trúc này.

Không có dữ liệu hoặc trọng số huấn luyện được nhúng vào notebook. Việc đạt 90% cần được xác nhận sau khi chạy.

Tham khảo: [ONNX Runtime quantization](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html),
[NCNN INT8](https://github.com/Tencent/ncnn/wiki/quantized-int8-inference),
[Multi-scale hand heatmaps](https://arxiv.org/abs/2105.10904).
Kiến trúc notebook là thiết kế encoder–decoder riêng, không phải bản tái hiện kết quả của bài báo.

Hướng dẫn BBB và benchmark nằm trong `BBB_DEPLOY.md` của ZIP. Chưa đo FPS thì chưa kết luận realtime.
